# M5: tiny T4 attention and NF4 compatibility check

CPU preparation already passed. This checks small library operations before model weights.

1. Import this file as a **new private Kaggle notebook**. **No input dataset is needed.**
2. Select **Internet On** and **GPU T4 x2** only when ready. Start **Run All within one minute** of GPU assignment. If the session has already been allocated longer, stop it and report the displayed elapsed time before running.
3. Run once. The only download is the verified **60.7 MB bitsandbytes wheel** into temporary storage. The base environment is unchanged. No model weights or generations run.
4. Download the result ZIP and the executed notebook with outputs, then **stop the GPU session immediately**. Save them under **`notebooks/run/6/`** and report the displayed allocation duration. Avoid Save & Run All: that can execute the work again.

The controller admits work within 360 seconds, including a 90-second export reserve. Allow 120 seconds for retrieval/stopping. With at most 60 seconds before the first cell, the conditional ceiling is 0.3 aggregate GPU-hour for two assigned T4s. Prior/setup/idle time still counts; this grants no fresh model-smoke allowance. Provider start/release is not inferred from cell timestamps.

Matching host Torch 2.11.0+cu128 is required; a mismatch exports diagnostics without changing packages. This standalone library environment is not the accepted scientific model lock. Tiny successful operations cannot pass full-context memory, Qwen loading/recurrent behavior or model smoke. The final cell exports failures too; return the first result rather than rerunning.


In [1]:
import hashlib,json,os,time
from pathlib import Path

if 'COMPATIBILITY_STARTED' not in globals():
    COMPATIBILITY_STARTED=time.monotonic()
OPERATIONS_PROGRAM='"""Bounded CPU source/environment preparation; never load weights or run models."""\n\nfrom __future__ import annotations\n\nfrom contextlib import closing\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport platform\nimport re\nimport shutil\nimport signal\nimport sqlite3\nimport stat\nimport subprocess\nimport sys\nimport time\nimport uuid\nimport zipfile\n\n\nPINNED = {"torch": "2.11.0+cu128", "pydantic": "2.13.5", "langgraph": "1.2.12",\n          "langgraph-checkpoint": "4.2.0", "langgraph-checkpoint-sqlite": "3.1.1",\n          "langchain-core": "1.6.6", "transformers": "5.12.0", "accelerate": "1.14.0",\n          "bitsandbytes": "0.49.2", "tokenizers": "0.22.2", "huggingface-hub": "1.19.0"}\nINDEX_ARGS = ["--index-url", "https://pypi.org/simple", "--extra-index-url",\n              "https://download.pytorch.org/whl/cu128"]\nOUTPUT_RESERVE = 256 * 2**20\nBUILD_RESERVE = 2**30\nCPU_SECONDS = 2700\nEXPORT_SECONDS = 90\nINVENTORY_LIMIT = 2000\nFOCUSED = ["tests.test_inference_preparation", "tests.inference_checks", "tests.durable_checks",\n           "tests.attempt_checks", "tests.worker_checks"]\nSTAGES = ["input_inventory", "source", "cpu_host", "command_cleanup", "environment", "installed_packages",\n          "focused_checks", "native_9b", "native_4b", "native_capacity", "filesystem_backup"]\nMODEL_SELECTED_BYTES = {"9B": 19319226111, "4B": 9332739901}\nPIP_WHEEL_RUNNER = ("import runpy,sys;sys.path.insert(0,sys.argv.pop(1));"\n                    "runpy.run_module(\'pip\',run_name=\'__main__\',alter_sys=True)")\n\n\ndef utc() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef sha256_file(path: Path) -> str:\n    with path.open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef save_json(path: Path, value: object) -> None:\n    temporary = path.with_suffix(path.suffix + ".partial")\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef inventory(root: Path, limit: int = INVENTORY_LIMIT) -> dict:\n    entries, errors, pending = [], [], [root]\n    truncated = False\n    if root.is_symlink() or not root.is_dir():\n        raise ValueError(f"input_root_missing_or_symlink: {root}")\n    while pending:\n        directory = pending.pop()\n        try:\n            with os.scandir(directory) as children:\n                for child in children:\n                    if len(entries) >= limit:\n                        truncated = True\n                        break\n                    info = child.stat(follow_symlinks=False)\n                    kind = ("symlink" if child.is_symlink() else "directory" if stat.S_ISDIR(info.st_mode)\n                            else "file" if stat.S_ISREG(info.st_mode) else "other")\n                    entries.append({"path": Path(child.path).relative_to(root).as_posix(),\n                                    "kind": kind, "bytes": info.st_size})\n                    if kind == "directory":\n                        pending.append(Path(child.path))\n        except OSError as exc:\n            errors.append({"path": str(directory), "reason": str(exc)})\n        if truncated:\n            break\n    return {"root": str(root), "entries": sorted(entries, key=lambda x: x["path"]),\n            "limit": limit, "truncated": truncated, "errors": errors}\n\n\ndef transformed_sources(root: Path, listing: dict) -> list[dict]:\n    entries = {x["path"]: x for x in listing["entries"]}\n    result = []\n    for name, item in entries.items():\n        if item["kind"] != "file" or not name.endswith("SOURCE_COMMIT.txt") or item["bytes"] > 128:\n            continue\n        source = PurePosixPath(name).parent\n        paths = {}\n        for fixture in ("capacity_report.json.gz", "capacity_report.json",\n                        "scripted_lifecycle_run.zip", "scripted_lifecycle_run"):\n            relative = (source / "tests/fixtures" / fixture).as_posix()\n            paths[fixture] = entries.get(relative, {"kind": "missing"})\n        result.append({"root": str(root / source), "paths": paths})\n    return result\n\n\ndef safe_members(archive: zipfile.ZipFile, expected: dict) -> list[zipfile.ZipInfo]:\n    infos = archive.infolist()\n    names = []\n    for entry in infos:\n        name = entry.orig_filename\n        if (not name or name != entry.filename or "\\\\" in name or ":" in name\n                or name.startswith("/") or any(ord(c) < 32 for c in name)\n                or any(part in {"", ".", ".."} for part in name.split("/"))\n                or entry.is_dir() or stat.S_ISLNK(entry.external_attr >> 16)):\n            raise ValueError(f"unsafe_archive_member: {name!r}")\n        if name in names or any(str(p) in names for p in PurePosixPath(name).parents):\n            raise ValueError(f"duplicate_or_conflicting_member: {name}")\n        if any(other.startswith(name + "/") for other in names):\n            raise ValueError(f"conflicting_member: {name}")\n        names.append(name)\n    if set(names) != set(expected):\n        raise ValueError(f"archive_member_set: missing={sorted(set(expected)-set(names))}; "\n                         f"unexpected={sorted(set(names)-set(expected))}")\n    for entry in infos:\n        if entry.file_size != expected[entry.filename]["bytes"]:\n            raise ValueError(f"archive_member_size: {entry.filename}")\n    return infos\n\n\ndef admit_source(payload: Path, destination: Path, expected: dict) -> dict:\n    if expected["expanded_bytes"] != sum(x["bytes"] for x in expected["members"].values()):\n        raise ValueError("source_manifest_expanded_bytes")\n    if payload.is_symlink() or not payload.is_file():\n        raise ValueError("payload_not_regular")\n    if payload.stat().st_size != expected["archive_bytes"]:\n        raise ValueError("payload_size_mismatch")\n    if sha256_file(payload) != expected["archive_sha256"]:\n        raise ValueError("payload_hash_mismatch")\n    if destination.exists():\n        raise FileExistsError("source_destination_exists")\n    with zipfile.ZipFile(payload) as archive:\n        infos = safe_members(archive, expected["members"])\n        # Validate every member before executing/importing/copying any source.\n        for entry in infos:\n            with archive.open(entry) as stream:\n                digest = hashlib.file_digest(stream, "sha256").hexdigest()\n            if digest != expected["members"][entry.filename]["sha256"]:\n                raise ValueError(f"archive_member_hash: {entry.filename}")\n        marker = archive.read("SOURCE_COMMIT.txt").decode("ascii").strip()\n        if marker != expected["source_baseline_commit"]:\n            raise ValueError("source_baseline_marker")\n        destination.mkdir(parents=True, exist_ok=False)\n        for entry in infos:\n            target = destination / entry.filename\n            if not target.resolve().is_relative_to(destination.resolve()):\n                raise ValueError(f"source_path_escape: {entry.filename}")\n            target.parent.mkdir(parents=True, exist_ok=True)\n            with archive.open(entry) as incoming, target.open("xb") as output:\n                shutil.copyfileobj(incoming, output, 1024 * 1024)\n            if sha256_file(target) != expected["members"][entry.filename]["sha256"]:\n                raise ValueError(f"extracted_member_hash: {entry.filename}")\n    return {"payload": str(payload), "archive_sha256": expected["archive_sha256"],\n            "verified_members": len(infos), "expanded_bytes": expected["expanded_bytes"],\n            "baseline_commit": marker, "build_commit": expected.get("build_commit"),\n            "dataset_version": None, "dataset_reference": str(payload.parent),\n            "closure": "Every canonical archive member, including unchanged compressed fixtures."}\n\n\ndef filesystem_facts(paths: dict[str, Path]) -> dict:\n    locations, pools = {}, {}\n    for role, path in paths.items():\n        info, disk = path.stat(), shutil.disk_usage(path)\n        key = str(info.st_dev)\n        locations[role] = {"path": str(path), "pool": key, "free_bytes": disk.free,\n                           "total_bytes": disk.total}\n        pools[key] = {"free_bytes": min(disk.free, pools.get(key, {}).get("free_bytes", disk.free)),\n                      "roles": pools.get(key, {}).get("roles", []) + [role]}\n    return {"locations": locations, "pools": pools, "quota_verified": False,\n            "interpretation": "Filesystem free space is observed; shared pools are counted once."}\n\n\ndef require_space(scratch: Path, output: Path, additional_scratch: int) -> dict:\n    facts = filesystem_facts({"scratch": scratch, "output": output})\n    same = scratch.stat().st_dev == output.stat().st_dev\n    if shutil.disk_usage(output).free < OUTPUT_RESERVE:\n        raise ValueError("output_evidence_reserve")\n    required = additional_scratch + (OUTPUT_RESERVE if same else 0)\n    scratch_pool=facts[\'locations\'][\'scratch\'][\'pool\']\n    if facts[\'pools\'][scratch_pool][\'free_bytes\'] < required:\n        raise ValueError(f"scratch_capacity: additional_required={required}")\n    facts.update(additional_scratch_bytes=additional_scratch, output_reserve_bytes=OUTPUT_RESERVE)\n    return facts\n\n\ndef _group_exists(pid: int, proc_root: Path = Path(\'/proc\')) -> bool:\n    try:\n        os.killpg(pid, 0)\n    except ProcessLookupError:\n        return False\n    if not proc_root.is_dir():\n        return True  # Presence is known; live membership cannot be inspected.\n    for path in proc_root.iterdir():\n        if not path.name.isdigit():\n            continue\n        try:\n            fields=(path/\'stat\').read_text().rsplit(\')\',1)[1].split()\n            if int(fields[2])==pid and fields[0] not in {\'Z\',\'X\'}:\n                return True\n        except FileNotFoundError:\n            continue  # Process exited during the read.\n        except (OSError,ValueError,IndexError):\n            return True  # Never claim cleanup from an incomplete inspection.\n    return False  # Zombies have exited and hold no executable/CUDA state.\n\n\ndef _stop_owned(process: subprocess.Popen) -> dict:\n    if os.name == "nt":\n        result = subprocess.run(["taskkill", "/PID", str(process.pid), "/T", "/F"],\n                                capture_output=True, timeout=5)\n        try:\n            process.wait(timeout=3)\n        except subprocess.TimeoutExpired:\n            process.kill()\n            process.wait(timeout=3)\n        return {"leader_exited": process.poll() is not None,\n                "tree_confirmation": "taskkill" if result.returncode == 0 else "unconfirmed"}\n    for sig in (signal.SIGTERM, signal.SIGKILL):\n        if not _group_exists(process.pid):\n            break\n        try:\n            os.killpg(process.pid, sig)\n        except ProcessLookupError:\n            break\n        end = time.monotonic() + 2\n        while time.monotonic() < end:\n            process.poll()\n            if not _group_exists(process.pid):\n                break\n            time.sleep(0.05)\n    process.wait(timeout=2)\n    return {"leader_exited": True,\n            "tree_confirmation": "exited" if not _group_exists(process.pid) else "unconfirmed"}\n\n\ndef command(output: Path, name: str, args: list, *, deadline: float, timeout: float,\n            cwd: Path, env: dict) -> dict:\n    remaining = deadline - time.monotonic() - EXPORT_SECONDS\n    if remaining < timeout + 7:\n        raise TimeoutError(f"deadline_before_command: {name}, remaining={remaining:.1f}, bound={timeout}")\n    logs = output / "logs"\n    logs.mkdir(exist_ok=True)\n    stdout, stderr = logs / (name + ".stdout.txt"), logs / (name + ".stderr.txt")\n    record = {"name": name, "args": list(map(str, args)), "cwd": str(cwd), "started_utc": utc(),\n              "timeout_seconds": timeout, "stdout": stdout.relative_to(output).as_posix(),\n              "stderr": stderr.relative_to(output).as_posix(), "status":"failed",\n              "leader_exited":False,"tree_confirmation":"not_started"}\n    started, process = time.monotonic(), None\n    try:\n        with stdout.open("xb") as out, stderr.open("xb") as err:\n            options = {"creationflags": subprocess.CREATE_NEW_PROCESS_GROUP} if os.name == "nt" else {\n                "start_new_session": True}\n            process = subprocess.Popen(list(map(str, args)), cwd=cwd, env=env, stdout=out, stderr=err,\n                                       stdin=subprocess.DEVNULL, **options)\n            record["pid"] = process.pid\n            record[\'tree_confirmation\']=\'unconfirmed\'\n            try:\n                record["exit_code"] = process.wait(timeout=timeout)\n                record["status"] = "passed" if process.returncode == 0 else "failed"\n                record["leader_exited"] = True\n                if os.name != "nt" and _group_exists(process.pid):\n                    record.update(_stop_owned(process), status="failed", reason="descendants_survived")\n                elif os.name != "nt":\n                    record[\'tree_confirmation\']=\'exited\'\n                else:\n                    record[\'tree_confirmation\']=\'not_checked_on_windows\'\n            except BaseException as exc:\n                record.update(status="failed", reason=type(exc).__name__, **_stop_owned(process))\n                raise\n    finally:\n        record.update(ended_utc=utc(), elapsed_seconds=time.monotonic()-started)\n        save_json(logs / (name + ".json"), record)\n    if record["status"] != "passed":\n        raise RuntimeError(f"command_failed: {name}, exit={record.get(\'exit_code\')}, "\n                           f"reason={record.get(\'reason\')}; see logs/{name}.stderr.txt")\n    return record\n\n\ndef json_stdout(output: Path, record: dict) -> object:\n    path = output / record["stdout"]\n    if path.stat().st_size > 2 * 2**20:\n        raise ValueError(f"machine_stdout_too_large: {record[\'name\']}")\n    return json.loads(path.read_text(encoding="utf-8"))\n\n\ndef process_tree_diagnostic(output: Path, scratch: Path, deadline: float, env: dict) -> dict:\n    ticker=scratch/\'process-tree-ticks\'\n    grandchild="import pathlib,time;p=pathlib.Path("+repr(str(ticker))+");[(p.write_text(str(n)),time.sleep(.05)) for n in range(600)]"\n    child="import subprocess,sys,time;subprocess.Popen([sys.executable,\'-c\',"+repr(grandchild)+"]);time.sleep(30)"\n    parent="import subprocess,sys,time;subprocess.Popen([sys.executable,\'-c\',"+repr(child)+"]);print(\'owned-tree-started\',flush=True);time.sleep(30)"\n    try:\n        command(output,\'owned-process-timeout\',[sys.executable,\'-c\',parent],deadline=deadline,\n                timeout=2,cwd=scratch,env=env)\n    except subprocess.TimeoutExpired:\n        record=json.loads((output/\'logs/owned-process-timeout.json\').read_text())\n    else:\n        raise ValueError(\'owned_tree_expected_timeout_missing\')\n    confirmation=\'taskkill\' if os.name==\'nt\' else \'exited\'\n    if not record.get(\'leader_exited\') or record.get(\'tree_confirmation\')!=confirmation or not ticker.is_file():\n        raise ValueError(\'owned_tree_cleanup_unconfirmed\')\n    before=ticker.read_text()\n    time.sleep(.2)\n    if ticker.read_text()!=before:\n        raise ValueError(\'owned_grandchild_still_running\')\n    return {\'expected_timeout\':True,\'leader_exited\':True,\'tree_confirmation\':confirmation,\n            \'grandchild_stopped\':True,\'scope\':\'Synthetic CPU process diagnostic; no model or GPU initialization.\'}\n\n\ndef test_result(output: Path, record: dict, expected_count: int) -> dict:\n    text = (output / record["stderr"]).read_text(encoding="utf-8")\n    matches = re.findall(r"^Ran (\\d+) tests? in ([\\d.]+)s$", text, flags=re.MULTILINE)\n    ok = re.search(r"^OK(?: \\(([^\\n]+)\\))?\\s*$", text, flags=re.MULTILINE)\n    skipped = re.search(r"skipped=(\\d+)", ok.group(1) or "") if ok else None\n    if len(matches) != 1 or not ok or int(matches[0][0]) != expected_count or skipped:\n        raise ValueError(f"test_count_or_skip: {record[\'name\']}, expected={expected_count}")\n    return {"tests": expected_count, "skipped": 0, "reported_seconds": float(matches[0][1]),\n            "scope": "Source-mode candidate-host checks; no physical model generation."}\n\n\ndef sqlite_diagnostic(output: Path) -> dict:\n    database, backup = output / "filesystem.sqlite", output / "filesystem-backup.sqlite"\n    with closing(sqlite3.connect(database, timeout=1)) as db:\n        journal = db.execute("PRAGMA journal_mode=WAL").fetchone()[0]\n        db.execute("PRAGMA synchronous=FULL")\n        synchronous = db.execute("PRAGMA synchronous").fetchone()[0]\n        db.execute("CREATE TABLE probe(value INTEGER NOT NULL)")\n        db.execute("INSERT INTO probe VALUES (7)")\n        db.commit()\n        with closing(sqlite3.connect(backup, timeout=1)) as copy:\n            db.backup(copy)\n    with closing(sqlite3.connect(backup.as_uri()+"?mode=ro&immutable=1", uri=True, timeout=1)) as db:\n        integrity = db.execute("PRAGMA integrity_check").fetchone()[0]\n        rows = db.execute("SELECT value FROM probe").fetchall()\n        try:\n            db.execute("INSERT INTO probe VALUES (8)")\n        except sqlite3.OperationalError:\n            read_only = True\n        else:\n            read_only = False\n    if journal != "wal" or synchronous != 2 or integrity != "ok" or rows != [(7,)] or not read_only:\n        raise ValueError("filesystem_sqlite_diagnostic")\n    return {"journal_mode": journal, "synchronous": synchronous, "integrity": integrity,\n            "rows": rows, "read_only_enforced": read_only, "backup_sha256": sha256_file(backup),\n            "scope": "Small synthetic diagnostic; actual worker/crash checks are separate."}\n\n\ndef candidate_environment(scratch: Path) -> dict:\n    env = {k: v for k, v in os.environ.items()\n           if not k.startswith(("PYTHON", "PIP_", "HF_", "TRANSFORMERS_", "VIRTUAL_ENV",\n                                "LANGSMITH_", "LANGCHAIN_"))}\n    env.update(PYTHONNOUSERSITE="1", PYTHONDONTWRITEBYTECODE="1", PYTHONUTF8="1",\n               TOKENIZERS_PARALLELISM="false", HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",\n               HF_HUB_CACHE=str(scratch/"hub"), TMPDIR=str(scratch/"tmp"), TEMP=str(scratch/"tmp"),\n               TMP=str(scratch/"tmp"), PIP_CONFIG_FILE=os.devnull)\n    (scratch/"tmp").mkdir(exist_ok=True)\n    return env\n\n\ndef pip_wheel_download_script(metadata: dict, destination: Path) -> str:\n    """One pinned bootstrap wheel, downloaded by an owned, bounded stdlib process."""\n    return "\\n".join([\n        "import hashlib,json", "from pathlib import Path",\n        "from urllib.parse import urlsplit", "from urllib.request import urlopen",\n        f"expected={metadata!r}", f"destination=Path({str(destination)!r})",\n        "def checked_origin(url):",\n        "    parts=urlsplit(url)",\n        "    if (parts.scheme!=\'https\' or parts.hostname!=\'files.pythonhosted.org\'",\n        "            or parts.port not in (None,443) or parts.username or parts.password):",\n        "        raise ValueError(\'pip_bootstrap_origin\')",\n        "checked_origin(expected[\'url\'])",\n        "if destination.name!=expected[\'filename\'] or expected[\'bytes\']<=0:",\n        "    raise ValueError(\'pip_bootstrap_metadata\')",\n        "if destination.exists(): raise FileExistsError(destination)",\n        "partial=destination.with_suffix(destination.suffix+\'.partial\')",\n        "digest=hashlib.sha256();size=0",\n        "with urlopen(expected[\'url\'],timeout=30) as response:",\n        "    checked_origin(response.geturl())",\n        "    with partial.open(\'xb\') as stream:",\n        "        while data:=response.read(1024*1024):",\n        "            size+=len(data)",\n        "            if size>expected[\'bytes\']: raise ValueError(\'pip_bootstrap_size\')",\n        "            digest.update(data);stream.write(data)",\n        "if size!=expected[\'bytes\']: raise ValueError(\'pip_bootstrap_size\')",\n        "if digest.hexdigest()!=expected[\'sha256\']: raise ValueError(\'pip_bootstrap_hash\')",\n        "partial.replace(destination)",\n        "print(json.dumps(dict(expected,verified=True)))", ""])\n\n\ndef installed_probe_script(expected: dict, versions: dict, result_path: Path) -> str:\n    hashes = {n: item["sha256"] for n, item in expected["members"].items()\n              if n.startswith("recovery/") and n.endswith(".py")}\n    return "\\n".join([\n        "import hashlib,json,sys,site", "from pathlib import Path",\n        "from importlib.metadata import version", "import recovery,recovery.inference,recovery.runner",\n        "from langgraph.checkpoint.sqlite import SqliteSaver",\n        "from transformers import Qwen3_5ForCausalLM", "import torch",\n        f"wanted={versions!r}", "actual={n:version(n) for n in wanted}",\n        "assert actual==wanted, {\'expected\':wanted,\'actual\':actual}",\n        "assert sys.prefix!=sys.base_prefix and site.ENABLE_USER_SITE is False",\n        "assert torch.__version__==\'2.11.0+cu128\' and torch.version.cuda==\'12.8\'",\n        "package=Path(recovery.__file__).resolve().parent", f"expected={hashes!r}",\n        "observed={\'recovery/\'+p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in package.glob(\'*.py\')}",\n        "assert observed==expected, \'source-to-installed module bytes differ\'",\n        "assert package.is_relative_to(Path(sys.prefix).resolve()), \'base/source import contamination\'",\n        f"Path({str(result_path)!r}).write_text(json.dumps({{\'versions\':actual,\'executable\':sys.executable,\'prefix\':sys.prefix,",\n        "\'sys_path\':sys.path,\'module_root\':str(package),\'verified_module_hashes\':observed,",\n        "\'qwen_text_class\':Qwen3_5ForCausalLM.__name__,\'torch_cuda_build\':torch.version.cuda,",\n        "\'gpu_native_diagnostics\':None,\'model_generations\':0}),encoding=\'utf-8\')", ""])\n\n\ndef installed_provenance(result_path: Path, expected: dict, versions: dict) -> dict:\n    # Third-party imports may print diagnostics; machine evidence has its own file.\n    if result_path.stat().st_size > 2*2**20:\n        raise ValueError(\'installed_probe_result_too_large\')\n    result = json.loads(result_path.read_text(encoding=\'utf-8\'))\n    wanted_hashes = {n:p[\'sha256\'] for n,p in expected[\'members\'].items()\n                     if n.startswith(\'recovery/\') and n.endswith(\'.py\')}\n    if (not isinstance(result,dict) or result.get(\'versions\')!=versions\n            or result.get(\'verified_module_hashes\')!=wanted_hashes\n            or result.get(\'qwen_text_class\')!=\'Qwen3_5ForCausalLM\'\n            or result.get(\'torch_cuda_build\')!=\'12.8\' or result.get(\'model_generations\')!=0):\n        raise ValueError(\'installed_probe_schema_or_provenance\')\n    return result\n\n\ndef export_evidence(output: Path) -> dict:\n    paths = sorted(p for p in output.rglob("*") if p.is_file() and not p.is_symlink()\n                   and (p.suffix in {".json", ".txt", ".lock", ".sqlite"}))\n    partial, final = output/"preflight-results.zip.partial", output/"preflight-results.zip"\n    with zipfile.ZipFile(partial, "x", compression=zipfile.ZIP_DEFLATED) as archive:\n        for path in paths:\n            archive.write(path, path.relative_to(output).as_posix())\n    with zipfile.ZipFile(partial) as archive:\n        if archive.testzip() is not None or archive.namelist() != [p.relative_to(output).as_posix() for p in paths]:\n            raise ValueError("export_crc_or_members")\n    partial.replace(final)\n    return {"zip_path": str(final), "zip_sha256": sha256_file(final), "zip_bytes": final.stat().st_size}\n\n\ndef run_preflight(input_root: Path, output_root: Path, scratch_root: Path, expected: dict,\n                  lock_text: str, *, workflow_seconds: float = CPU_SECONDS) -> dict:\n    started = time.monotonic()\n    deadline = started + workflow_seconds\n    label = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")+"-"+uuid.uuid4().hex[:8]\n    output = output_root/(\'recovery-m5-preflight-\'+label)\n    output.mkdir(parents=True, exist_ok=False)\n    scratch = scratch_root/label\n    source, candidate = scratch/"source", scratch/"venv"\n    report = {"schema_version": 2, "kind": "cpu-canonical-source-preparation", "started_utc": utc(),\n              "output_directory": str(output), "scratch_directory": str(scratch),\n              "cpu_workflow_limit_seconds": workflow_seconds, "export_reserve_seconds": EXPORT_SECONDS,\n              "stages": {n: {"status": "not_run", "reason": None} for n in STAGES}, "root_blocker": None}\n    payload, listing, env = None, None, None\n\n    def stage(name, prerequisites, action):\n        state = report["stages"][name]\n        state["prerequisites"] = prerequisites\n        blocked = [p for p in prerequisites if report["stages"][p]["status"] != "passed"]\n        if blocked:\n            state.update(status="blocked", reason="prerequisites: "+", ".join(blocked))\n        elif deadline-time.monotonic() < EXPORT_SECONDS:\n            state.update(status="blocked", reason="cpu_deadline_before_stage")\n            report["root_blocker"] = report["root_blocker"] or "cpu_deadline_before_stage"\n        else:\n            then = time.monotonic()\n            state.update(started_utc=utc(), status="running")\n            save_json(output/"preflight-report.json", report)\n            try:\n                state.update(result=action(), status="passed")\n            except KeyboardInterrupt:\n                state.update(status="failed", reason="user_interruption")\n                report["root_blocker"] = report["root_blocker"] or "user_interruption"\n                raise\n            except Exception as exc:\n                state.update(status="failed", reason=f"{type(exc).__name__}: {exc}")\n                report["root_blocker"] = report["root_blocker"] or state["reason"]\n            finally:\n                state.update(ended_utc=utc(), elapsed_seconds=time.monotonic()-then)\n        save_json(output/"preflight-report.json", report)\n        print(f"{name}: {state[\'status\']}"+(f" — {state[\'reason\']}" if state["reason"] else ""), flush=True)\n\n    def input_stage():\n        nonlocal payload, listing\n        listing = inventory(input_root)\n        save_json(output/"input-inventory.json", listing)\n        save_json(output/"source-input-diagnosis.json", {"transformed_sources": transformed_sources(input_root, listing)})\n        if listing["truncated"] or listing["errors"]:\n            raise ValueError("incomplete_input_inventory")\n        choices = [input_root/x["path"] for x in listing["entries"]\n                   if x["kind"] == "file" and PurePosixPath(x["path"]).name == expected["payload_filename"]]\n        if len(choices) != 1:\n            raise ValueError("payload_missing" if not choices else "payload_ambiguous")\n        payload = choices[0]\n        return {"selected_payload": str(payload), "entries": len(listing["entries"])}\n\n    def source_stage():\n        scratch.mkdir(parents=True, exist_ok=False)\n        facts = require_space(scratch, output, expected["expanded_bytes"]+BUILD_RESERVE)\n        save_json(output/"storage-before-source.json", facts)\n        return admit_source(payload, source, expected)\n\n    def host_stage():\n        gpu_dir = Path("/proc/driver/nvidia/gpus")\n        gpus = sorted(p.name for p in gpu_dir.iterdir()) if gpu_dir.is_dir() else []\n        host = {"python": platform.python_version(), "os": platform.platform(),\n                "sqlite": sqlite3.sqlite_version, "visible_driver_gpu_entries": gpus,\n                "cuda_initialized_by_kernel": False}\n        save_json(output/"host.json", host)\n        if gpus:\n            raise ValueError("cpu_stage_gpu_visible: stop session and select Accelerator None")\n        if platform.system() != "Linux" or sys.version_info[:2] != (3, 13) or platform.machine() != "x86_64":\n            raise ValueError("candidate_host_requires_linux_x86_64_python313")\n        return host\n\n    def invoke(name, args, timeout, *, cwd=None, source_mode=False):\n        if report.get(\'process_cleanup_unconfirmed\'):\n            raise RuntimeError(\'prior_process_cleanup_unconfirmed\')\n        require_space(scratch, output, BUILD_RESERVE)\n        process_env = dict(env)\n        if source_mode:\n            process_env["PYTHONPATH"] = str(source)\n        try:\n            return command(output, name, args, deadline=deadline, timeout=timeout,\n                           cwd=cwd or scratch, env=process_env)\n        except BaseException:\n            record_path = output/\'logs\'/(name+\'.json\')\n            if record_path.exists() and json.loads(record_path.read_text()).get(\'tree_confirmation\')==\'unconfirmed\':\n                report[\'process_cleanup_unconfirmed\']=True\n            raise\n\n    interpreter = candidate/("Scripts/python.exe" if os.name == "nt" else "bin/python")\n\n    def environment_stage():\n        nonlocal env\n        if hashlib.sha256(lock_text.encode()).hexdigest() != expected["candidate_lock_sha256"]:\n            raise ValueError("candidate_lock_hash")\n        env = candidate_environment(scratch)\n        lock = scratch/"candidate.lock"\n        lock.write_text(lock_text, encoding="utf-8")\n        (output/"candidate.lock").write_text(lock_text, encoding="utf-8")\n        versions = dict(re.findall(r"^([\\w.-]+)==([^\\s\\\\]+)", lock_text, flags=re.MULTILINE))\n        if any(versions.get(n) != v for n,v in PINNED.items()):\n            raise ValueError("candidate_roots_changed")\n        # Run 3 failed inside ensurepip; create the same plain venv without invoking it.\n        invoke("create-venv", [sys.executable, "-m", "venv", "--without-pip", candidate], 60)\n        configuration = (candidate/"pyvenv.cfg").read_text().lower()\n        if "include-system-site-packages = false" not in configuration:\n            raise ValueError("candidate_inherits_base_packages")\n        tools = expected["build_tools"]\n        files = expected["candidate_wheels"]\n        if {p[\'name\']:p[\'version\'] for p in files} != versions:\n            raise ValueError("candidate_wheel_closure")\n        download_bytes = sum(p["bytes"] for p in files)\n        # This stated package-expansion forecast is replaced by exact wheel ZIP sizes below.\n        facts = require_space(scratch, output, 4*download_bytes+BUILD_RESERVE)\n        facts.update(download_bytes=download_bytes, package_expansion_forecast_multiplier=4)\n        save_json(output/"storage-before-wheels.json", facts)\n        save_json(output/"wheel-files.json", files)\n        bootstrap = scratch/"pip-bootstrap.lock"\n        p = tools["pip"]\n        pip_metadata = p[\'wheels\'][0]\n        pip_directory = scratch/\'pip-bootstrap\'\n        pip_directory.mkdir()\n        pip_wheel = pip_directory/pip_metadata[\'filename\']\n        downloaded = json_stdout(output, invoke(\'download-pip-wheel\', [interpreter,\'-I\',\'-c\',\n                     pip_wheel_download_script(pip_metadata,pip_wheel)], 60))\n        if (downloaded != dict(pip_metadata,verified=True) or not pip_wheel.is_file()\n                or pip_wheel.stat().st_size != pip_metadata[\'bytes\']\n                or sha256_file(pip_wheel) != pip_metadata[\'sha256\']):\n            raise ValueError(\'pip_bootstrap_unverified_wheel\')\n        save_json(output/\'pip-bootstrap-wheel.json\', downloaded)\n        bootstrap.write_text(f"pip=={p[\'version\']} --hash=sha256:{pip_metadata[\'sha256\']}\\n")\n        invoke("bootstrap-pip", [interpreter,"-I","-c",PIP_WHEEL_RUNNER,pip_wheel,\n               "--isolated","--disable-pip-version-check","install","--no-cache-dir",\n               "--no-deps","--require-hashes","--report",output/"pip-bootstrap.json",\n               "--no-index","--find-links",pip_directory,"-r",bootstrap], 120)\n        pip = [interpreter,"-m","pip","--isolated","--disable-pip-version-check"]\n        common = ["--no-cache-dir","--only-binary=:all:","--require-hashes","-r",lock]\n        wheels = scratch/"wheels"\n        wheels.mkdir()\n        direct_lock = scratch/"candidate-download.lock"\n        direct_lock.write_text("".join(f"{p[\'name\']} @ {p[\'url\']} --hash=sha256:{p[\'sha256\']}\\n" for p in files))\n        invoke("download-candidate", pip+["download","--no-deps","--dest",wheels,"--no-cache-dir",\n               "--require-hashes","--only-binary=:all:","-r",direct_lock],900)\n        tool_lock = scratch/"build-tools.lock"\n        tool_lock.write_text("".join(f"{n}=={tools[n][\'version\']} --hash=sha256:{tools[n][\'wheels\'][0][\'sha256\']}\\n"\n                                    for n in ("setuptools","wheel")))\n        invoke("download-build-tools", pip+["download","--no-deps","--dest",wheels,"--only-binary=:all:",\n               "--require-hashes","--no-cache-dir","-r",tool_lock,"--index-url","https://pypi.org/simple"],120)\n        expected_files = {p[\'filename\']:p for p in files}\n        expected_files.update({tools[n][\'wheels\'][0][\'filename\']:tools[n][\'wheels\'][0] for n in (\'setuptools\',\'wheel\')})\n        actual_files = {p.name:p for p in wheels.iterdir()}\n        if set(actual_files) != set(expected_files):\n            raise ValueError("wheel_file_set")\n        expanded = 0\n        for name,path in actual_files.items():\n            wanted = expected_files[name]\n            if path.stat().st_size != wanted[\'bytes\'] or sha256_file(path) != wanted[\'sha256\']:\n                raise ValueError(f"wheel_bytes: {name}")\n            with zipfile.ZipFile(path) as archive:\n                expanded += sum(p.file_size for p in archive.infolist())\n        save_json(output/"storage-before-install.json", require_space(scratch,output,expanded+BUILD_RESERVE))\n        invoke("resolve-candidate",pip+["install","--dry-run","--ignore-installed","--no-index",\n               "--find-links",wheels,"--report",output/"resolver.json"]+common,120)\n        resolver = json.loads((output/"resolver.json").read_text())\n        resolved = {p["metadata"]["name"].lower().replace("_","-"): p["metadata"]["version"]\n                    for p in resolver["install"]}\n        if resolved != versions:\n            raise ValueError("actual_resolver_closure_differs_from_candidate")\n        invoke("install-candidate",pip+["install","--no-index","--find-links",wheels,"--no-compile",\n               "--report",output/"install.json"]+common,300)\n        constraints = scratch/"build-constraints.txt"\n        constraints.write_text("".join(f"{n}=={tools[n][\'version\']}\\n" for n in (\'setuptools\',\'wheel\')))\n        built = scratch/"built"\n        built.mkdir()\n        invoke("build-source",pip+["wheel","--no-deps","--no-index","--find-links",wheels,\n               "--build-constraint",constraints,"--wheel-dir",built,source,"-v"],120)\n        built_files = list(built.glob(\'*.whl\'))\n        if len(built_files) != 1:\n            raise ValueError("source_wheel_count")\n        save_json(output/"source-wheel.json", {"filename":built_files[0].name,\n                  "bytes":built_files[0].stat().st_size,"sha256":sha256_file(built_files[0]),\n                  "build_tools":{n:tools[n][\'version\'] for n in (\'setuptools\',\'wheel\')}})\n        invoke("install-source",pip+["install","--no-deps","--no-index","--no-compile",\n               "--report",output/"source-install.json",built_files[0]],60)\n        return {"mode":"clean-venv-offline-wheel-install","versions":versions,\n                "resolved_packages":len(versions),"download_bytes":download_bytes,\n                "wheel_expanded_bytes":expanded,"build_tools":tools}\n\n    def packages_stage():\n        invoke("pip-check", [interpreter,"-m","pip","--isolated","check"],30)\n        probe = scratch/"installed_probe.py"\n        versions = report[\'stages\'][\'environment\'][\'result\'][\'versions\']\n        result_path = output/\'installed-probe-result.json\'\n        probe.write_text(installed_probe_script(expected,versions,result_path),encoding=\'utf-8\')\n        invoke("installed-probe", [interpreter,"-I",probe],90)\n        result = installed_provenance(result_path,expected,versions)\n        save_json(output/"installed-provenance.json",result)\n        record = invoke("pip-list", [interpreter,"-m","pip","--isolated","list","--format=json"],30)\n        save_json(output/"installed-packages.json",json_stdout(output,record))\n        freeze = invoke("pip-freeze", [interpreter,"-m","pip","--isolated","freeze","--all"],30)\n        shutil.copyfile(output/freeze[\'stdout\'],output/\'installed-freeze.txt\')\n        return result\n\n    def tests_stage(name,args,count,bound):\n        record = invoke(name,[interpreter]+args,bound,cwd=source,source_mode=True)\n        return test_result(output,record,count)\n\n    result = {"zip_path":None,"zip_sha256":None,"run_directory":str(output)}\n    try:\n        save_json(output/"expected-source-manifest.json",expected)\n        stage("input_inventory",[],input_stage)\n        stage("source",["input_inventory"],source_stage)\n        stage("cpu_host",[],host_stage)\n        stage("command_cleanup",["source","cpu_host"],lambda:process_tree_diagnostic(output,scratch,deadline,candidate_environment(scratch)))\n        stage("environment",["source","cpu_host","command_cleanup"],environment_stage)\n        stage("installed_packages",["environment"],packages_stage)\n        stage("focused_checks",["installed_packages"],lambda:tests_stage("focused-checks",["-m","unittest",*FOCUSED,"-q"],81,360))\n        stage("native_9b",["installed_packages"],lambda:tests_stage("native-9b",["-m","tests.tokenizer_inference","--tokenizer","metadata/9B"],4,60))\n        stage("native_4b",["installed_packages"],lambda:tests_stage("native-4b",["-m","tests.tokenizer_inference","--tokenizer","metadata/4B"],4,60))\n        stage("native_capacity",["installed_packages"],lambda:tests_stage("native-capacity",["-m","tests.tokenizer_capacity","--tokenizer","metadata/9B"],8,90))\n        stage("filesystem_backup",[],lambda:sqlite_diagnostic(output))\n    except KeyboardInterrupt:\n        report[\'root_blocker\'] = report[\'root_blocker\'] or \'user_interruption\'\n        for state in report[\'stages\'].values():\n            if state[\'status\'] in {\'running\',\'not_run\'}:\n                state.update(status=\'blocked\',reason=\'user_interruption\')\n    finally:\n        states = report[\'stages\']\n        passed = all(s[\'status\']==\'passed\' for s in states.values())\n        tests = sum(s.get(\'result\',{}).get(\'tests\',0) for s in states.values() if s[\'status\']==\'passed\')\n        summary = {"preparation_checks_passed":passed,"root_blocker":report[\'root_blocker\'],\n                   "tests_passed":tests,"model_generations":0,"model_smoke_ready":False,\n                   "gpu_native_diagnostics":None,"model_assets_verified":None,"model_smoke":None,\n                   "provider_assignment_utc":None,"provider_release_utc":None,\n                   "observed_cpu_wall_seconds":time.monotonic()-started,\n                   "next_action":("Download the ZIP and stop the session. Return it for candidate-host review before GPU/model work."\n                    if passed else "Download the ZIP and stop the session. Return the root-blocker evidence for review; do not rerun or repair fixtures.")}\n        if report[\'root_blocker\'] and \'payload_missing\' in report[\'root_blocker\']:\n            summary[\'next_action\']=\'Attach the private dataset containing unchanged m5-kaggle-source.payload. Download this ZIP, stop the session, and return the evidence.\'\n        report[\'summary\']=summary\n        save_json(output/\'preflight-report.json\',report)\n        save_json(output/\'summary.json\',summary)\n        try:\n            if scratch.exists():\n                storage=filesystem_facts({\'scratch\':scratch,\'output\':output})\n                storage[\'models\']={n:{\'selected_file_bytes\':b,\'assets_staged\':False,\'quota_admitted\':False}\n                                   for n,b in MODEL_SELECTED_BYTES.items()}\n                save_json(output/\'storage-after.json\',storage)\n        except Exception as exc:\n            report[\'storage_after_error\']=f\'{type(exc).__name__}: {exc}\'\n            save_json(output/\'preflight-report.json\',report)\n        try:\n            result.update(export_evidence(output))\n        except Exception as exc:\n            report[\'export_error\']=f\'{type(exc).__name__}: {exc}\'\n            save_json(output/\'preflight-report.json\',report)\n        result.update(summary=summary,report=report)\n    return result\n'
COMPATIBILITY_PROGRAM='"""Bounded standalone Torch/NF4 diagnostics; no pretrained models or generation."""\n\nfrom __future__ import annotations\n\nimport argparse\nfrom contextlib import nullcontext\nimport csv\nimport hashlib\nfrom importlib import metadata\nimport io\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport platform\nimport re\nimport shutil\nimport stat\nimport sys\nimport time\nimport uuid\nimport warnings\nimport zipfile\n\nWORKFLOW_SECONDS = 360\nPRESTART_SECONDS = 60\nSTOP_SECONDS = 120\nGPU_COUNT = 2\nTORCH_VERSION = "2.11.0+cu128"\nATTENTION_CASES = (\n    ("prefill-gqa", "prefill", False, False, False),\n    ("prefill-expanded", "prefill", True, False, False),\n    ("prefill-expanded-efficient", "prefill", True, False, True),\n    ("prefill-mask-expanded", "prefill", True, True, False),\n    ("decode-gqa", "decode", False, False, False),\n    ("decode-expanded", "decode", True, False, False),\n    ("decode-expanded-efficient", "decode", True, False, True),\n)\n\n\ndef read_machine(path: Path) -> dict:\n    if path.stat().st_size > 2 * 2**20:\n        raise ValueError("machine_result_too_large")\n    result = json.loads(path.read_text(encoding="utf-8"))\n    if not isinstance(result, dict):\n        raise ValueError("machine_result_not_object")\n    return result\n\n\ndef envelope_assessment(configs: dict) -> dict:\n    models = {}\n    for name, item in configs.items():\n        raw = item["text"].encode("utf-8")\n        if hashlib.sha256(raw).hexdigest() != item["sha256"]:\n            raise ValueError("model_config_hash: " + name)\n        outer = json.loads(raw)\n        c = outer["text_config"]\n        layers = c["layer_types"]\n        if (c["num_attention_heads"], c["num_key_value_heads"], c["head_dim"],\n                c["num_hidden_layers"], len(layers), layers.count("full_attention")) != (16, 4, 256, 32, 32, 8):\n            raise ValueError("reviewed_attention_geometry_changed")\n        tied = c.get("tie_word_embeddings", outer["tie_word_embeddings"])\n        if type(tied) is not bool:\n            raise ValueError("invalid_tie_setting")\n        kv = 2 * layers.count("full_attention") * c["num_key_value_heads"] * c["head_dim"] * 2\n        embeddings = c["vocab_size"] * c["hidden_size"] * 2 * (1 if tied else 2)\n        mlp = 3 * c["hidden_size"] * c["intermediate_size"] * c["num_hidden_layers"] // 2\n        resident = 262144\n        models[name] = {\n            "config_sha256": item["sha256"], "tie_word_embeddings": tied,\n            "kv_bytes_per_token": kv, "resident_token_assumption": resident,\n            "fp16_kv_bytes_at_cap": kv * resident,\n            "conditional_embedding_head_bytes": embeddings,\n            "ideal_packed_mlp_bytes": mlp,\n            "conditional_subtotal_bytes_at_cap": embeddings + mlp + kv * resident,\n            "repeated_kv_one_layer_additional_bytes_at_cap": 2 * c["num_attention_heads"] * c["head_dim"] * 2 * resident,\n            "dense_score_fp32_bytes_at_cap": c["num_attention_heads"] * resident * resident * 4,\n            "full_envelope_memory_acceptance": None,\n        }\n    if set(models) != {"9B", "4B"}:\n        raise ValueError("expected_both_pinned_configs")\n    return {"models": models, "full_envelope_memory_acceptance": None,\n            "scope": "Conditional arithmetic; excludes other weights/scales/activations/workspaces. Legal attainability and actual model memory remain unknown."}\n\n\ndef parse_inventory(text: str) -> list[dict]:\n    devices = []\n    for row in csv.reader(io.StringIO(text)):\n        if len(row) != 7:\n            raise ValueError("nvidia_inventory_columns")\n        index, identity, name, pci, driver, total, free = [v.strip() for v in row]\n        if not re.fullmatch(r"GPU-[0-9a-fA-F-]{36}", identity) or not name.endswith("T4"):\n            raise ValueError("expected_physical_T4_uuid")\n        uuid.UUID(identity[4:])\n        devices.append({"index": int(index), "uuid": identity, "name": name, "pci_bus_id": pci,\n                        "driver_version": driver, "total_bytes": int(total) * 2**20,\n                        "free_bytes": int(free) * 2**20})\n    if len(devices) != GPU_COUNT or len({d["uuid"].lower() for d in devices}) != GPU_COUNT or len({d["index"] for d in devices}) != GPU_COUNT:\n        raise ValueError("expected_two_distinct_T4s")\n    if any(d["free_bytes"] < 1536 * 2**20 or d["total_bytes"] < d["free_bytes"] for d in devices):\n        raise ValueError("insufficient_free_memory_for_tiny_probe")\n    return sorted(devices, key=lambda d: d["index"])\n\n\ndef unpack_wheel(wheel: Path, destination: Path, expected: dict, ops: dict, output: Path) -> dict:\n    if wheel.stat().st_size != expected["bytes"] or ops["sha256_file"](wheel) != expected["sha256"]:\n        raise ValueError("bitsandbytes_wheel_bytes")\n    with zipfile.ZipFile(wheel) as archive:\n        infos = archive.infolist()\n        if len({m.filename.casefold() for m in infos}) != len(infos):\n            raise ValueError("wheel_duplicate_paths")\n        expanded = sum(m.file_size for m in infos)\n        if expanded != expected["expanded_bytes"] or len(infos) != expected["members"] or archive.testzip() is not None:\n            raise ValueError("wheel_expansion_or_crc")\n        for member in infos:\n            path = PurePosixPath(member.filename)\n            if (member.orig_filename != member.filename or path.is_absolute() or\n                    any(p in {".", ".."} for p in path.parts) or "\\\\" in member.filename or\n                    ":" in member.filename or any(ord(c) < 32 for c in member.filename) or\n                    stat.S_ISLNK(member.external_attr >> 16) or\n                    path.parts[0] not in {"bitsandbytes", "bitsandbytes-0.49.2.dist-info"}):\n                raise ValueError("unsafe_wheel_path: " + member.filename)\n        ops["require_space"](wheel.parent, output, expanded + 2**20)\n        destination.mkdir(exist_ok=False)\n        files = {}\n        for member in infos:\n            path = destination / member.filename\n            if not path.resolve().is_relative_to(destination.resolve()):\n                raise ValueError("wheel_path_escape")\n            if member.is_dir():\n                path.mkdir(parents=True, exist_ok=True)\n                continue\n            path.parent.mkdir(parents=True, exist_ok=True)\n            with archive.open(member) as incoming, path.open("xb") as outgoing:\n                shutil.copyfileobj(incoming, outgoing, 2**20)\n            files[member.filename] = {"bytes": path.stat().st_size, "sha256": ops["sha256_file"](path)}\n    return {"wheel": expected, "files": files, "expanded_bytes": expanded}\n\n\ndef backend_name(operators: list[str]) -> str | None:\n    names = {backend for backend, marker in {\n        "math": "aten::_scaled_dot_product_attention_math",\n        "efficient": "aten::_scaled_dot_product_efficient_attention",\n        "flash": "aten::_scaled_dot_product_flash_attention",\n        "cudnn": "aten::_scaled_dot_product_cudnn_attention",\n    }.items() if marker in operators}\n    return next(iter(names)) if len(names) == 1 else None\n\n\ndef attention_inputs(torch, phase: str):\n    if phase not in {"prefill", "decode"}:\n        raise ValueError("unknown_diagnostic_phase")\n    generator = torch.Generator(device="cpu").manual_seed(813)\n    query = torch.randn(1, 16, 32 if phase == "prefill" else 1, 256, generator=generator) * 0.25\n    key = torch.randn(1, 4, 32, 256, generator=generator) * 0.25\n    value = torch.randn(1, 4, 32, 256, generator=generator) * 0.25\n    return query, key, value\n\n\ndef sdpa_comparison(torch, device, case: tuple) -> dict:\n    from torch.nn.attention import SDPBackend, sdpa_kernel\n    from torch.profiler import ProfilerActivity, profile\n\n    name, phase, expanded, masked, forced = case\n    query, key, value = attention_inputs(torch, phase)\n    mask = torch.ones(query.shape[2], key.shape[2], dtype=torch.bool).tril() if masked else None\n    causal = phase == "prefill" and mask is None\n    with torch.inference_mode(), sdpa_kernel(SDPBackend.MATH):\n        reference = torch.nn.functional.scaled_dot_product_attention(\n            query, key.repeat_interleave(4, dim=1), value.repeat_interleave(4, dim=1),\n            attn_mask=mask, dropout_p=0.0, is_causal=causal)\n    q, k, v = [t.to(device=device, dtype=torch.float16) for t in (query, key, value)]\n    if expanded:\n        k, v = k.repeat_interleave(4, dim=1), v.repeat_interleave(4, dim=1)\n    mask = mask.to(device) if mask is not None else None\n    eligible = {}\n    with warnings.catch_warnings(record=True) as captured:\n        warnings.simplefilter("always")\n        if device.type == "cuda":\n            params = torch.backends.cuda.SDPAParams(q, k, v, mask, 0.0, causal, not expanded)\n            for backend, function in [("flash", "can_use_flash_attention"),\n                                      ("efficient", "can_use_efficient_attention"),\n                                      ("cudnn", "can_use_cudnn_attention")]:\n                eligible[backend] = bool(getattr(torch.backends.cuda, function)(params, debug=True))\n        activities = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if device.type == "cuda" else [])\n        started = time.monotonic()\n        try:\n            with torch.inference_mode(), sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION) if forced else nullcontext():\n                with profile(activities=activities) as observed:\n                    result = torch.nn.functional.scaled_dot_product_attention(\n                        q, k, v, attn_mask=mask, dropout_p=0.0, is_causal=causal, enable_gqa=not expanded)\n                    if device.type == "cuda":\n                        torch.cuda.synchronize()\n            operators = sorted({event.key for event in observed.key_averages()})\n            kernels = sorted({event.name for event in observed.events() if str(event.device_type).endswith("CUDA")})\n            actual = result.float().cpu()\n            finite = bool(torch.isfinite(actual).all())\n            close = bool(torch.allclose(actual, reference, rtol=0.02, atol=0.002))\n            if not finite or not close:\n                raise ValueError("nonfinite_or_mismatched_sdpa: " + name)\n            observation = {"status": "passed", "finite": finite, "reference_matches": close,\n                           "max_absolute_error": float((actual - reference).abs().max()),\n                           "operator_backend": backend_name(operators), "operators": operators,\n                           "cuda_kernels": kernels, "cuda_kernel_trace_available": bool(kernels)}\n        except RuntimeError as exc:\n            if not forced or not any(s in str(exc) for s in ["No available kernel", "No viable backend"]):\n                raise\n            observation = {"status": "unsupported", "reason": str(exc), "operator_backend": None}\n    observation.update(name=name, phase=phase, query_shape=list(q.shape), key_shape=list(k.shape),\n                       value_shape=list(v.shape), dtype=str(q.dtype), mask_shape=list(mask.shape) if mask is not None else None,\n                       is_causal=causal, enable_gqa=not expanded, forced_efficient=forced,\n                       eligibility=eligible, warnings=[str(w.message) for w in captured],\n                       elapsed_seconds=time.monotonic()-started, rtol=0.02, atol=0.002)\n    return observation\n\n\ndef nf4_comparison(torch, bnb, device) -> dict:\n    generator = torch.Generator(device="cpu").manual_seed(814)\n    weight = (torch.randn(128, 256, generator=generator) * 0.1).to(device=device, dtype=torch.float16)\n    packed, state = bnb.functional.quantize_4bit(weight, blocksize=64, compress_statistics=True, quant_type="nf4")\n    if state.quant_type != "nf4" or not state.nested or state.dtype != torch.float16:\n        raise ValueError("nf4_double_quant_fp16_settings")\n    dequantized = bnb.functional.dequantize_4bit(packed, quant_state=state).float().cpu()\n    rows = []\n    with torch.inference_mode():\n        for batch in (8, 1):\n            inputs = (torch.randn(batch, 256, generator=generator) * 0.1).to(device=device, dtype=torch.float16)\n            actual = bnb.matmul_4bit(inputs, packed.t(), quant_state=state)\n            expected = inputs.float().cpu() @ dequantized.t()\n            compared = actual.float().cpu()\n            finite = bool(torch.isfinite(compared).all())\n            close = bool(torch.allclose(compared, expected, rtol=0.02, atol=0.01))\n            if actual.dtype != torch.float16 or list(actual.shape) != [batch, 128] or not finite or not close:\n                raise ValueError("nonfinite_or_mismatched_nf4")\n            rows.append({"batch": batch, "output_shape": list(actual.shape), "output_dtype": str(actual.dtype),\n                         "finite": finite, "reference_matches": close,\n                         "max_absolute_error": float((compared-expected).abs().max())})\n    return {"status": "passed", "weight_shape": [128, 256], "quant_type": state.quant_type,\n            "double_quantization": bool(state.nested), "weight_dtype": str(state.dtype),\n            "blocksize": state.blocksize, "rtol": 0.02, "atol": 0.01, "operations": rows,\n            "scope": "Synthetic operator check against the same dequantized weight; no pretrained model."}\n\n\ndef gpu_probe(output: Path, package_root: Path, snapshot_path: Path, identity: str) -> None:\n    report = {"kind": "standalone-t4-library-probe", "status": "failed", "assigned_uuid": identity,\n              "model_generations": 0, "attention": {}}\n\n    def save():\n        output.write_text(json.dumps(report, indent=2)+"\\n", encoding="utf-8")\n\n    save()\n    try:\n        snapshot = read_machine(snapshot_path)\n        actual_paths = {p.relative_to(package_root).as_posix() for p in package_root.rglob("*") if p.is_file()}\n        if actual_paths != set(snapshot["files"]):\n            raise ValueError("changed_extracted_wheel_file_set")\n        for name, item in snapshot["files"].items():\n            path = package_root / name\n            if path.is_symlink() or path.stat().st_size != item["bytes"] or hashlib.sha256(path.read_bytes()).hexdigest() != item["sha256"]:\n                raise ValueError("changed_extracted_wheel: " + name)\n        if os.environ.get("CUDA_VISIBLE_DEVICES") != identity or os.environ.get("BNB_CUDA_VERSION"):\n            raise ValueError("device_or_native_library_override")\n        sys.path.insert(0, str(package_root))\n        import torch\n        if torch.__version__ != TORCH_VERSION or torch.version.cuda != "12.8" or not torch.cuda.is_available():\n            raise ValueError("matching_cuda_torch_unavailable")\n        if torch.cuda.device_count() != 1:\n            raise ValueError("worker_requires_one_visible_device")\n        props = torch.cuda.get_device_properties(0)\n        observed_uuid = str(props.uuid)\n        if uuid.UUID(observed_uuid.removeprefix("GPU-")) != uuid.UUID(identity.removeprefix("GPU-")) or (props.major, props.minor) != (7, 5):\n            raise ValueError("physical_device_identity_or_capability")\n        torch.set_num_threads(1)\n        device = torch.device("cuda:0")\n        torch.cuda.reset_peak_memory_stats()\n        report.update(observed_uuid=observed_uuid, device_count=1, capability=[props.major, props.minor],\n                      device_name=props.name, total_memory=props.total_memory, torch_version=torch.__version__,\n                      torch_cuda_build=torch.version.cuda, torch_root=str(Path(torch.__file__).resolve()),\n                      torch_native_root=str(Path(torch._C.__file__).resolve()), compiled_architectures=torch.cuda.get_arch_list(),\n                      torch_configuration=torch.__config__.show(), pre_probe_free_bytes=torch.cuda.mem_get_info()[0])\n        save()\n        for case in ATTENTION_CASES:\n            print("Running " + case[0], flush=True)\n            report["attention"][case[0]] = sdpa_comparison(torch, device, case)\n            save()\n        import bitsandbytes as bnb\n        if metadata.version("bitsandbytes") != "0.49.2" or not Path(bnb.__file__).resolve().is_relative_to(package_root.resolve()):\n            raise ValueError("bitsandbytes_import_origin")\n        library = Path(bnb.cextension.lib._lib._name).resolve()\n        if not library.is_relative_to(package_root.resolve()) or library.name != "libbitsandbytes_cuda128.so":\n            raise ValueError("bitsandbytes_native_library_origin")\n        report.update(bitsandbytes_version=metadata.version("bitsandbytes"), bitsandbytes_root=str(Path(bnb.__file__).resolve()),\n                      bitsandbytes_native_library=str(library))\n        print("Running NF4 double-quantized FP16 operations", flush=True)\n        report["nf4"] = nf4_comparison(torch, bnb, device)\n        torch.cuda.synchronize()\n        report.update(status="passed", peak_allocated_bytes=torch.cuda.max_memory_allocated(),\n                      peak_reserved_bytes=torch.cuda.max_memory_reserved(), post_probe_free_bytes=torch.cuda.mem_get_info()[0],\n                      full_candidate_gpu_acceptance=None, native_qwen_recurrent_acceptance=None,\n                      full_envelope_memory_acceptance=None)\n    except BaseException as exc:\n        report["error"] = f"{type(exc).__name__}: {exc}"\n        raise\n    finally:\n        save()\n\n\ndef validate_probe(result: dict, identity: str, package_root: Path) -> None:\n    if (result.get("status") != "passed" or result.get("assigned_uuid") != identity or\n            uuid.UUID(result["observed_uuid"].removeprefix("GPU-")) != uuid.UUID(identity.removeprefix("GPU-")) or\n            result.get("device_count") != 1 or result.get("capability") != [7, 5] or\n            result.get("torch_version") != TORCH_VERSION or result.get("torch_cuda_build") != "12.8" or\n            result.get("bitsandbytes_version") != "0.49.2" or type(result.get("model_generations")) is not int or\n            result.get("model_generations") != 0):\n        raise ValueError("gpu_probe_identity_or_scope")\n    if (not Path(result["bitsandbytes_root"]).resolve().is_relative_to(package_root.resolve()) or\n            not Path(result["bitsandbytes_native_library"]).resolve().is_relative_to(package_root.resolve()) or\n            Path(result["bitsandbytes_native_library"]).name != "libbitsandbytes_cuda128.so" or\n            any(result.get(k) is not None for k in ["full_candidate_gpu_acceptance", "native_qwen_recurrent_acceptance", "full_envelope_memory_acceptance"])):\n        raise ValueError("probe_import_origin_or_acceptance_scope")\n    if set(result["attention"]) != {c[0] for c in ATTENTION_CASES}:\n        raise ValueError("missing_attention_case")\n    for name, phase, expanded, masked, forced in ATTENTION_CASES:\n        row = result["attention"][name]\n        if (row["query_shape"] != [1, 16, 32 if phase == "prefill" else 1, 256] or\n                row["key_shape"] != [1, 16 if expanded else 4, 32, 256] or row["value_shape"] != row["key_shape"] or\n                row["enable_gqa"] is not (not expanded) or row["is_causal"] is not (phase == "prefill" and not masked) or\n                row.get("dtype") != "torch.float16" or row.get("mask_shape") != ([32, 32] if masked else None) or\n                row.get("forced_efficient") is not forced):\n            raise ValueError("invalid_attention_observation: " + name)\n        if row["status"] == "unsupported" and forced and any(s in row.get("reason", "") for s in ["No available kernel", "No viable backend"]):\n            continue\n        if row["status"] != "passed" or row.get("finite") is not True or row.get("reference_matches") is not True:\n            raise ValueError("failed_attention_observation: " + name)\n    nf4 = result["nf4"]\n    if (nf4.get("status") != "passed" or nf4.get("quant_type") != "nf4" or\n            nf4.get("double_quantization") is not True or nf4.get("weight_shape") != [128, 256] or\n            nf4.get("weight_dtype") != "torch.float16" or nf4.get("blocksize") != 64 or [r["batch"] for r in nf4["operations"]] != [8, 1] or\n            any(r.get("finite") is not True or r.get("reference_matches") is not True or\n                r.get("output_shape") != [r["batch"], 128] or r.get("output_dtype") != "torch.float16" for r in nf4["operations"])):\n        raise ValueError("invalid_nf4_observation")\n\n\ndef run_compatibility(output_root: Path, scratch_root: Path, expected: dict, ops: dict,\n                      program: str, started: float | None = None) -> dict:\n    started = time.monotonic() if started is None else started\n    deadline = started + WORKFLOW_SECONDS\n    label = uuid.uuid4().hex[:12]\n    output = output_root / ("recovery-m5-t4-compatibility-" + label)\n    output.mkdir(parents=True, exist_ok=False)\n    scratch = scratch_root / label\n    scratch.mkdir(parents=True, exist_ok=False)\n    stages = {name: {"status": "blocked", "reason": "earlier stage incomplete"} for name in\n              ["static_envelope", "host_inventory", "wheel_staging", "device_0", "device_1", "filesystem_backup"]}\n    report = {"kind": "standalone-t4-library-compatibility", "started_utc": ops["utc"](),\n              "controller_limit_seconds": WORKFLOW_SECONDS, "stages": stages, "root_blocker": None,\n              "process_cleanup_unconfirmed": False, "model_generations": 0}\n    result = {"run_directory": str(output), "zip_path": None, "zip_sha256": None, "report": report}\n    env = ops["candidate_environment"](scratch)\n    env.pop("BNB_CUDA_VERSION", None)\n    current, devices = "static_envelope", []\n    ops["save_json"](output / "expected-inputs.json", expected)\n\n    def invoke(name, args, timeout, identity=None):\n        if report["process_cleanup_unconfirmed"]:\n            raise RuntimeError("prior_cleanup_unconfirmed")\n        local_env = dict(env)\n        if identity:\n            local_env.update(CUDA_VISIBLE_DEVICES=identity, CUDA_DEVICE_ORDER="PCI_BUS_ID")\n        print("Running " + name + " ...", flush=True)\n        try:\n            return ops["command"](output, name, args, deadline=deadline, timeout=timeout, cwd=scratch, env=local_env)\n        finally:\n            log = output / "logs" / (name + ".json")\n            if log.is_file() and read_machine(log).get("tree_confirmation") == "unconfirmed":\n                report["process_cleanup_unconfirmed"] = True\n\n    def passed(value):\n        stages[current] = {"status": "passed", "result": value}\n        ops["save_json"](output / "compatibility-report.json", report)\n        print(current + ": passed", flush=True)\n\n    try:\n        passed(envelope_assessment(expected["configs"]))\n        current = "host_inventory"\n        software = {}\n        for name in expected["candidate_versions"]:\n            try:\n                software[name] = metadata.version(name)\n            except metadata.PackageNotFoundError:\n                software[name] = None\n        ops["save_json"](output / "host-software.json", {"python": platform.python_version(),\n            "os": platform.platform(), "versions": software, "candidate_differences": {\n                n: {"candidate": v, "host": software[n]} for n, v in expected["candidate_versions"].items() if software[n] != v},\n            "scope": "Host software observations, not the isolated accepted CPU environment or a scientific lock."})\n        if platform.system() != "Linux" or platform.machine() != "x86_64" or sys.version_info[:2] != (3, 13):\n            raise ValueError("diagnostic_requires_linux_x86_64_python313")\n        if software.get("torch") != TORCH_VERSION:\n            raise ValueError("host_torch_differs_from_reviewed_build")\n        record = invoke("gpu-inventory", ["nvidia-smi", "--query-gpu=index,uuid,name,pci.bus_id,driver_version,memory.total,memory.free",\n                                          "--format=csv,noheader,nounits"], 10)\n        devices = parse_inventory((output / record["stdout"]).read_text(encoding="utf-8"))\n        passed({"devices": devices, "provider_assignment_utc": None})\n        current = "wheel_staging"\n        wheel_meta = expected["bitsandbytes_wheel"]\n        ops["save_json"](output / "storage-before-wheel.json", ops["require_space"](\n            scratch, output, wheel_meta["bytes"] + wheel_meta["expanded_bytes"] + 2**20))\n        wheel = scratch / wheel_meta["filename"]\n        receipt = ops["json_stdout"](output, invoke("download-bitsandbytes-wheel", [sys.executable, "-I", "-B", "-c",\n                          ops["pip_wheel_download_script"](wheel_meta, wheel)], 60))\n        if receipt != dict(wheel_meta, verified=True):\n            raise ValueError("unverified_wheel_download_receipt")\n        package_root = scratch / "diagnostic-package"\n        snapshot = unpack_wheel(wheel, package_root, wheel_meta, ops, output)\n        snapshot_path = output / "extracted-wheel.json"\n        ops["save_json"](snapshot_path, snapshot)\n        passed({"verified_wheel_bytes": wheel.stat().st_size, "expanded_bytes": snapshot["expanded_bytes"],\n                "verified_files": len(snapshot["files"]), "base_environment_modified": False})\n        child = scratch / "gpu-probe.py"\n        child.write_text(program, encoding="utf-8")\n        (output / "gpu-probe-program.txt").write_text(program, encoding="utf-8")\n        for index, device in enumerate(devices):\n            current = "device_" + str(index)\n            machine = output / (current + ".json")\n            invoke(current, [sys.executable, "-I", "-B", child, "--probe", "--output", machine,\n                             "--package-root", package_root, "--snapshot", snapshot_path, "--uuid", device["uuid"]], 60, device["uuid"])\n            observed = read_machine(machine)\n            validate_probe(observed, device["uuid"], package_root)\n            passed(observed)\n    except BaseException as exc:\n        reason = f"{type(exc).__name__}: {exc}"\n        stages[current] = {"status": "failed", "reason": reason}\n        report["root_blocker"] = reason\n        print(current + ": failed - " + reason, flush=True)\n    finally:\n        try:\n            stages["filesystem_backup"] = {"status": "passed", "result": ops["sqlite_diagnostic"](output)}\n        except Exception as exc:\n            stages["filesystem_backup"] = {"status": "failed", "reason": f"{type(exc).__name__}: {exc}"}\n            report["root_blocker"] = report["root_blocker"] or stages["filesystem_backup"]["reason"]\n        elapsed = time.monotonic() - started\n        summary = {"library_checks_completed": all(s["status"] == "passed" for s in stages.values()),\n                   "root_blocker": report["root_blocker"], "model_generations": 0, "model_smoke_ready": False,\n                   "full_candidate_gpu_acceptance": None, "native_qwen_recurrent_acceptance": None,\n                   "full_envelope_memory_acceptance": None, "provider_assignment_utc": None, "provider_release_utc": None,\n                   "observed_controller_wall_seconds": elapsed, "observed_gpu_seconds_lower_bound": len(devices) * elapsed if devices else None,\n                   "timing_scope": "First code cell to the pre-export report; ZIP publication and provider stopping are outside this observed lower bound.",\n                   "conditional_allocation_upper_bound_seconds": GPU_COUNT * (PRESTART_SECONDS + WORKFLOW_SECONDS + STOP_SECONDS),\n                   "allocation_bounds_basis": "User must start within 60 seconds of assignment and stop within the 120-second reserve. Provider interval is unverified; prior allocations remain separate.",\n                   "attention_dispatch": {name: {case: {"status": row["status"], "backend": row.get("operator_backend")}\n                        for case, row in stage.get("result", {}).get("attention", {}).items()}\n                        for name, stage in stages.items() if name.startswith("device_")},\n                   "next_action": "Download this ZIP and the executed notebook; stop the GPU session immediately. Return both under notebooks/run/6/ and report displayed allocation duration. Do not rerun or start model smoke."}\n        report.update(summary=summary, ended_utc=ops["utc"]())\n        ops["save_json"](output / "compatibility-report.json", report)\n        ops["save_json"](output / "summary.json", summary)\n        result["summary"] = summary\n        try:\n            result.update(ops["export_evidence"](output))\n        except Exception as exc:\n            report["export_error"] = f"{type(exc).__name__}: {exc}"\n            ops["save_json"](output / "compatibility-report.json", report)\n        print(json.dumps(summary, indent=2), flush=True)\n    return result\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--probe", action="store_true", required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--package-root", type=Path, required=True)\n    parser.add_argument("--snapshot", type=Path, required=True)\n    parser.add_argument("--uuid", required=True)\n    args = parser.parse_args()\n    gpu_probe(args.output, args.package_root, args.snapshot, args.uuid)\n'
EXPECTED=json.loads('{"bitsandbytes_wheel": {"bytes": 60651714, "expanded_bytes": 184269948, "filename": "bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl", "members": 85, "name": "bitsandbytes", "sha256": "54b771f06e1a3c73af5c7f16ccf0fc23a846052813d4b008d10cb6e017dd1c8c", "url": "https://files.pythonhosted.org/packages/19/57/3443d6f183436fbdaf5000aac332c4d5ddb056665d459244a5608e98ae92/bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl", "version": "0.49.2"}, "build_commit": "79ae1169e8a1a1e79c58171ce855535348b3db16", "candidate_versions": {"accelerate": "1.14.0", "aiosqlite": "0.22.1", "annotated-doc": "0.0.5", "annotated-types": "0.8.0", "anyio": "4.15.1", "bitsandbytes": "0.49.2", "certifi": "2026.7.22", "charset-normalizer": "3.5.2", "click": "8.5.0", "cuda-bindings": "12.9.9", "cuda-pathfinder": "1.8.3", "cuda-toolkit": "12.8.1", "distro": "1.9.0", "filelock": "4.0.12", "fsspec": "2026.9.0", "h11": "0.16.0", "hf-xet": "1.7.0", "httpcore": "1.0.9", "httpcore2": "2.13.1", "httpx": "0.28.1", "httpx2": "2.13.1", "huggingface-hub": "1.19.0", "idna": "3.20", "jinja2": "3.1.6", "jsonpatch": "1.33", "jsonpointer": "3.2.0", "langchain-core": "1.6.6", "langchain-protocol": "0.0.19", "langgraph": "1.2.12", "langgraph-checkpoint": "4.2.0", "langgraph-checkpoint-sqlite": "3.1.1", "langgraph-prebuilt": "1.1.0", "langgraph-sdk": "0.4.6", "langsmith": "0.14.4", "markdown-it-py": "4.2.0", "markupsafe": "3.0.4", "mdurl": "0.1.2", "mpmath": "1.3.0", "networkx": "3.7", "numpy": "2.5.3", "nvidia-cublas-cu12": "12.8.4.1", "nvidia-cuda-cupti-cu12": "12.8.90", "nvidia-cuda-nvrtc-cu12": "12.8.93", "nvidia-cuda-runtime-cu12": "12.8.90", "nvidia-cudnn-cu12": "9.19.0.56", "nvidia-cufft-cu12": "11.3.3.83", "nvidia-cufile-cu12": "1.13.1.3", "nvidia-curand-cu12": "10.3.9.90", "nvidia-cusolver-cu12": "11.7.3.90", "nvidia-cusparse-cu12": "12.5.8.93", "nvidia-cusparselt-cu12": "0.7.1", "nvidia-nccl-cu12": "2.28.9", "nvidia-nvjitlink-cu12": "12.8.93", "nvidia-nvshmem-cu12": "3.4.5", "nvidia-nvtx-cu12": "12.8.90", "orjson": "3.13.0", "ormsgpack": "1.12.2", "packaging": "26.3", "psutil": "7.2.2", "pydantic": "2.13.5", "pydantic-core": "2.46.5", "pygments": "2.21.0", "pyyaml": "6.0.3", "regex": "2026.9.29", "requests": "2.34.2", "requests-toolbelt": "1.0.0", "rich": "15.0.0", "safetensors": "0.8.0", "setuptools": "81.0.0", "shellingham": "1.5.4", "sniffio": "1.3.1", "sqlite-vec": "0.1.9", "sympy": "1.14.0", "tenacity": "9.2.1", "tokenizers": "0.22.2", "torch": "2.11.0+cu128", "tqdm": "4.70.1", "transformers": "5.12.0", "triton": "3.6.0", "truststore": "0.10.4", "typer": "0.25.1", "typing-extensions": "4.16.0", "typing-inspection": "0.4.4", "urllib3": "2.8.0", "uuid-utils": "0.17.1", "websockets": "16.1.1", "xxhash": "4.0.1", "zstandard": "0.25.0"}, "canonical_archive_sha256": "57f99508f47f5571769891e39603db0c2b15c99038e0a186f3e2d40a9d48622f", "configs": {"4B": {"sha256": "ddc63e1c717afa86c865bb5e01313d89d72bb53b97ad4a8a03ba8510c0621670", "text": "{\\n    \\"architectures\\": [\\n        \\"Qwen3_5ForConditionalGeneration\\"\\n    ],\\n    \\"image_token_id\\": 248056,\\n    \\"model_type\\": \\"qwen3_5\\",\\n    \\"text_config\\": {\\n        \\"attention_bias\\": false,\\n        \\"attention_dropout\\": 0.0,\\n        \\"attn_output_gate\\": true,\\n        \\"dtype\\": \\"bfloat16\\",\\n        \\"eos_token_id\\": 248044,\\n        \\"full_attention_interval\\": 4,\\n        \\"head_dim\\": 256,\\n        \\"hidden_act\\": \\"silu\\",\\n        \\"hidden_size\\": 2560,\\n        \\"initializer_range\\": 0.02,\\n        \\"intermediate_size\\": 9216,\\n        \\"layer_types\\": [\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"full_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"full_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"full_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"full_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"full_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"full_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"full_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"full_attention\\"\\n        ],\\n        \\"linear_conv_kernel_dim\\": 4,\\n        \\"linear_key_head_dim\\": 128,\\n        \\"linear_num_key_heads\\": 16,\\n        \\"linear_num_value_heads\\": 32,\\n        \\"linear_value_head_dim\\": 128,\\n        \\"max_position_embeddings\\": 262144,\\n        \\"mlp_only_layers\\": [],\\n        \\"model_type\\": \\"qwen3_5_text\\",\\n        \\"mtp_num_hidden_layers\\": 1,\\n        \\"mtp_use_dedicated_embeddings\\": false,\\n        \\"num_attention_heads\\": 16,\\n        \\"num_hidden_layers\\": 32,\\n        \\"num_key_value_heads\\": 4,\\n        \\"rms_norm_eps\\": 1e-06,\\n        \\"tie_word_embeddings\\": true,\\n        \\"use_cache\\": true,\\n        \\"vocab_size\\": 248320,\\n        \\"mamba_ssm_dtype\\": \\"float32\\",\\n        \\"rope_parameters\\": {\\n            \\"mrope_interleaved\\": true,\\n            \\"mrope_section\\": [\\n                11,\\n                11,\\n                10\\n            ],\\n            \\"rope_type\\": \\"default\\",\\n            \\"rope_theta\\": 10000000,\\n            \\"partial_rotary_factor\\": 0.25\\n        }\\n    },\\n    \\"tie_word_embeddings\\": true,\\n    \\"transformers_version\\": \\"4.57.0.dev0\\",\\n    \\"video_token_id\\": 248057,\\n    \\"vision_config\\": {\\n        \\"deepstack_visual_indexes\\": [],\\n        \\"depth\\": 24,\\n        \\"hidden_act\\": \\"gelu_pytorch_tanh\\",\\n        \\"hidden_size\\": 1024,\\n        \\"in_channels\\": 3,\\n        \\"initializer_range\\": 0.02,\\n        \\"intermediate_size\\": 4096,\\n        \\"model_type\\": \\"qwen3_5\\",\\n        \\"num_heads\\": 16,\\n        \\"num_position_embeddings\\": 2304,\\n        \\"out_hidden_size\\": 2560,\\n        \\"patch_size\\": 16,\\n        \\"spatial_merge_size\\": 2,\\n        \\"temporal_patch_size\\": 2\\n    },\\n    \\"vision_end_token_id\\": 248054,\\n    \\"vision_start_token_id\\": 248053\\n}"}, "9B": {"sha256": "d0883072e01861ed0b2d47be3c16c36a8e81c224c7ffaa310c6558fb3f932b05", "text": "{\\n    \\"architectures\\": [\\n        \\"Qwen3_5ForConditionalGeneration\\"\\n    ],\\n    \\"image_token_id\\": 248056,\\n    \\"model_type\\": \\"qwen3_5\\",\\n    \\"text_config\\": {\\n        \\"attention_bias\\": false,\\n        \\"attention_dropout\\": 0.0,\\n        \\"attn_output_gate\\": true,\\n        \\"dtype\\": \\"bfloat16\\",\\n        \\"eos_token_id\\": 248044,\\n        \\"full_attention_interval\\": 4,\\n        \\"head_dim\\": 256,\\n        \\"hidden_act\\": \\"silu\\",\\n        \\"hidden_size\\": 4096,\\n        \\"initializer_range\\": 0.02,\\n        \\"intermediate_size\\": 12288,\\n        \\"layer_types\\": [\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"full_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"full_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"full_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"full_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"full_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"full_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"full_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"linear_attention\\",\\n            \\"full_attention\\"\\n        ],\\n        \\"linear_conv_kernel_dim\\": 4,\\n        \\"linear_key_head_dim\\": 128,\\n        \\"linear_num_key_heads\\": 16,\\n        \\"linear_num_value_heads\\": 32,\\n        \\"linear_value_head_dim\\": 128,\\n        \\"max_position_embeddings\\": 262144,\\n        \\"mlp_only_layers\\": [],\\n        \\"model_type\\": \\"qwen3_5_text\\",\\n        \\"mtp_num_hidden_layers\\": 1,\\n        \\"mtp_use_dedicated_embeddings\\": false,\\n        \\"num_attention_heads\\": 16,\\n        \\"num_hidden_layers\\": 32,\\n        \\"num_key_value_heads\\": 4,\\n        \\"rms_norm_eps\\": 1e-06,\\n        \\"use_cache\\": true,\\n        \\"vocab_size\\": 248320,\\n        \\"mamba_ssm_dtype\\": \\"float32\\",\\n        \\"rope_parameters\\": {\\n            \\"mrope_interleaved\\": true,\\n            \\"mrope_section\\": [\\n                11,\\n                11,\\n                10\\n            ],\\n            \\"rope_type\\": \\"default\\",\\n            \\"rope_theta\\": 10000000,\\n            \\"partial_rotary_factor\\": 0.25\\n        }\\n    },\\n    \\"tie_word_embeddings\\": false,\\n    \\"transformers_version\\": \\"4.57.0.dev0\\",\\n    \\"video_token_id\\": 248057,\\n    \\"vision_config\\": {\\n        \\"deepstack_visual_indexes\\": [],\\n        \\"depth\\": 27,\\n        \\"hidden_act\\": \\"gelu_pytorch_tanh\\",\\n        \\"hidden_size\\": 1152,\\n        \\"in_channels\\": 3,\\n        \\"initializer_range\\": 0.02,\\n        \\"intermediate_size\\": 4304,\\n        \\"model_type\\": \\"qwen3_5\\",\\n        \\"num_heads\\": 16,\\n        \\"num_position_embeddings\\": 2304,\\n        \\"out_hidden_size\\": 4096,\\n        \\"patch_size\\": 16,\\n        \\"spatial_merge_size\\": 2,\\n        \\"temporal_patch_size\\": 2\\n    },\\n    \\"vision_end_token_id\\": 248054,\\n    \\"vision_start_token_id\\": 248053\\n}"}}, "operations_sha256": "cc28a9d43ff1fd5051c7e5bfd5fdcc1154a5d85f019ef1a00dda7f3ae5fceaf8", "program_sha256": "ae6f5f945f4df06a5f23b7274f4c0670afe90a9bc001251004740fdd86ac4567", "scientific_baseline_commit": "ca46bcd55486b38bb453f5294e01f943bc7f7752"}')
assert hashlib.sha256(OPERATIONS_PROGRAM.encode()).hexdigest()==EXPECTED['operations_sha256']
assert hashlib.sha256(COMPATIBILITY_PROGRAM.encode()).hexdigest()==EXPECTED['program_sha256']
operations_namespace={'__name__':'m5_compatibility_operations'}
exec(compile(OPERATIONS_PROGRAM,'m5_kaggle_preflight.py','exec'),operations_namespace)
compatibility_namespace={'__name__':'m5_t4_compatibility'}
exec(compile(COMPATIBILITY_PROGRAM,'m5_t4_compatibility.py','exec'),compatibility_namespace)


## Run the bounded library checks


In [2]:
# Path overrides are only for local validation; Kaggle defaults need no edits.
if 'COMPATIBILITY_RESULT' not in globals():
    COMPATIBILITY_RESULT=compatibility_namespace['run_compatibility'](
        Path(os.environ.get('M5_COMPAT_OUTPUT_ROOT','/kaggle/working')),
        Path(os.environ.get('M5_COMPAT_SCRATCH_ROOT','/tmp/recovery-m5-t4')),
        EXPECTED,operations_namespace,COMPATIBILITY_PROGRAM,COMPATIBILITY_STARTED)
else:
    print('Already executed in this kernel; showing the retained result without another run.')


static_envelope: passed
Running gpu-inventory ...
host_inventory: passed
Running download-bitsandbytes-wheel ...
wheel_staging: passed
Running device_0 ...
device_0: passed
Running device_1 ...
device_1: passed
{
  "library_checks_completed": true,
  "root_blocker": null,
  "model_generations": 0,
  "model_smoke_ready": false,
  "full_candidate_gpu_acceptance": null,
  "native_qwen_recurrent_acceptance": null,
  "full_envelope_memory_acceptance": null,
  "provider_assignment_utc": null,
  "provider_release_utc": null,
  "observed_controller_wall_seconds": 38.664189147,
  "observed_gpu_seconds_lower_bound": 77.328378294,
  "timing_scope": "First code cell to the pre-export report; ZIP publication and provider stopping are outside this observed lower bound.",
  "conditional_allocation_upper_bound_seconds": 1080,
  "allocation_bounds_basis": "User must start within 60 seconds of assignment and stop within the 120-second reserve. Provider interval is unverified; prior allocations remain se

## Download the evidence and stop


In [3]:
from IPython.display import FileLink,display

print(COMPATIBILITY_RESULT['summary']['next_action'])
if COMPATIBILITY_RESULT['zip_path']:
    print('Result ZIP SHA-256:',COMPATIBILITY_RESULT['zip_sha256'])
    display(FileLink(os.path.relpath(COMPATIBILITY_RESULT['zip_path'],Path.cwd())))
else:
    print('Export failed; retain JSON/logs from:',COMPATIBILITY_RESULT['run_directory'])
    print(COMPATIBILITY_RESULT['report'].get('export_error'))
print('Stop the GPU session now. Model smoke remains unexecuted.')


Download this ZIP and the executed notebook; stop the GPU session immediately. Return both under notebooks/run/6/ and report displayed allocation duration. Do not rerun or start model smoke.
Result ZIP SHA-256: 332237aa5b5b73b39bce42dc7ac98c7434bbea15148019b96dd615022c997a96


/kaggle/working/recovery-m5-t4-compatibility-b103e20420e4/preflight-results.zip

Stop the GPU session now. Model smoke remains unexecuted.
